# 05 — Window Functions

Ranking, offset, and aggregate window functions over partitions. Uses synthetic data with a department/employee shape.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Synthetic data

Employees with department and salary.

In [ ]:
import pandas as pd
from irispark import Window
from irispark.functions import col, row_number, rank, dense_rank, ntile, lag, lead, first_value, last_value, sum as s, avg

emp = session.createDataFrame(pd.DataFrame({
    "dept": ["Eng", "Eng", "Eng", "Sales", "Sales", "Sales"],
    "name": ["Ana", "Bruno", "Carlos", "Diana", "Eva", "Felipe"],
    "salary": [100.0, 200.0, 150.0, 300.0, 250.0, 200.0],
}))
emp.show()

## 2. Ranking functions

`row_number`, `rank`, `dense_rank`, `ntile` partitioned by department, ordered by salary descending.

In [ ]:
w = Window.partitionBy("dept").orderBy(col("salary").desc())

emp.withColumn("rn", row_number().over(w)) \
   .withColumn("rk", rank().over(w)) \
   .withColumn("dr", dense_rank().over(w)) \
   .withColumn("nt", ntile(2).over(w)) \
   .orderBy("dept", "rn") \
   .show()

## 3. Offset functions

`lag` and `lead` to compare a row with its neighbors.

In [ ]:
w = Window.partitionBy("dept").orderBy(col("salary").desc())

emp.withColumn("prev_salary", lag(col("salary"), 1).over(w)) \
   .withColumn("next_salary", lead(col("salary"), 1).over(w)) \
   .orderBy("dept", "salary DESC") \
   .show()

## 4. Value functions

`first_value` and `last_value`.

In [ ]:
w = Window.partitionBy("dept").orderBy(col("salary").desc())

emp.withColumn("top_salary", first_value(col("salary")).over(w)) \
   .withColumn("bottom_salary", last_value(col("salary")).over(w)) \
   .orderBy("dept", "salary DESC") \
   .show()

## 5. Aggregate window functions

Running totals and averages within a partition.

In [ ]:
w = Window.partitionBy("dept").orderBy(col("salary").desc())

emp.withColumn("running_total", s(col("salary")).over(w)) \
   .withColumn("dept_avg", avg(col("salary")).over(Window.partitionBy("dept"))) \
   .orderBy("dept", "salary DESC") \
   .show()

## 6. Frame bounds — `rowsBetween`

A 2-row sliding window.

In [ ]:
w = Window.partitionBy("dept").orderBy(col("salary").desc()).rowsBetween(-1, 0)

emp.withColumn("sliding_sum", s(col("salary")).over(w)) \
   .orderBy("dept", "salary DESC") \
   .show()

## 7. SQL transparency

In [ ]:
w = Window.partitionBy("dept").orderBy(col("salary").desc())
print(emp.withColumn("rn", row_number().over(w)).to_sql())

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")